In [ ]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Define the place to extract road network data
place_name = "Rio de Janeiro, Brazil"

# Download the road network graph (for vehicles)
graph = ox.graph_from_place(place_name, network_type="drive")

# Plot the road network
fig, ax = plt.subplots(figsize=(10, 10))
ox.plot_graph(
    graph,
    ax=ax,
    bgcolor="#ffffff",
    node_color="#1f77b4",
    node_size=0,
    edge_color="#1f77b4",
    edge_linewidth=0.2,
)
plt.show()

# Convert graph to GeoDataFrames
nodes, edges = ox.graph_to_gdfs(graph)

# Drop unnecessary columns from nodes and edges
nodes = nodes.drop(columns=["highway", "ref"])
edges = edges.drop(
    columns=[
        "maxspeed", "bridge", "ref", "access", "tunnel", "junction", "width",
        "area", "service", "osmid", "name", "highway", "reversed"
    ]
)

# Convert data types
nodes["street_count"] = nodes["street_count"].astype(float)
edges["oneway"] = edges["oneway"].astype(float)
edges["lanes"] = pd.to_numeric(edges["lanes"], errors="coerce").fillna(1).astype(float)

# Display the first few rows of nodes and edges
print(nodes.head().to_string())
print(edges.head().to_string())

# Reset the index of edges and drop the multi-index level ('key')
edges_reset = edges.droplevel("key").reset_index()

# Save nodes and edges to Shapefiles
nodes.to_file("/home/stagiaire/Téléchargements/TE3/osm_nodes.shp")
edges_reset.to_file("/home/stagiaire/Téléchargements/TE3/osm_edges.shp")

# Define OpenStreetMap tags for extracting industrial areas and other land uses
tags = {
    "landuse": [
        "industrial", "warehouse", "commercial", "retail", "factory",
        "construction", "grass", "cemetery", "railway", "recreation_ground",
        "farmland"
    ],
    "leisure": ["sports_centre", "recreation_ground"],
    "aeroway": "aerodrome",
    "building": "industrial",
}

# Extract geometries matching the specified tags
industrial_zones = ox.geometries_from_place(place_name, tags)

# Keep only polygons and multipolygons
industrial_zones_geom = gpd.GeoDataFrame(industrial_zones[["geometry"]])
industrial_zones_geom = industrial_zones_geom[
    industrial_zones_geom.geom_type.isin(["Polygon", "MultiPolygon"])
]

# Reproject to UTM Zone 23S (EPSG:32723)
industrial_zones_proj = industrial_zones_geom.to_crs(epsg=32723)

# Save the extracted industrial zones to a Shapefile
industrial_zones_proj.to_file("open_street_map.shp")
